# Day 18 · Colab 1 — Claude Agent + FastAPI Backend + Redis Memory

**Agentic Systems Bootcamp**

In this notebook you build a small but complete **tool-using agent**:

- a **FastAPI** backend exposing an `/orders/{id}` endpoint (your *external API*),
- a **Redis** memory layer (short-term conversation history + long-term facts),
- a **Claude** agent that calls tools in a `while` loop driven by `stop_reason`.

Everything runs **inside Colab** with no external servers:
`fakeredis` stands in for Redis and FastAPI's `TestClient` calls the API in-process.
Swap either for the real thing by changing one line each (shown at the end).

> **Pattern recap (from the deck):** client tools return `stop_reason="tool_use"`; *your* code runs the
> tool and returns a `tool_result` block; you loop until `stop_reason="end_turn"`.

## Step 1 — Install dependencies & set your API key

`fakeredis` gives us a real Redis API surface in-process. `httpx` is needed by FastAPI's test client.

In [1]:
!pip install -q anthropic fastapi 'httpx<0.28' fakeredis 2>/dev/null
print('deps installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.0/141.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 22.0 MB/s eta 0:00:00
deps installed


In [2]:
import os, getpass

# In Colab, prefer the Secrets panel (key icon) named ANTHROPIC_API_KEY.
if not os.environ.get('ANTHROPIC_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        pass

if not os.environ.get('ANTHROPIC_API_KEY'):
    # Fallback: paste it (hidden). Leave blank to run in OFFLINE mock mode.
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank = offline mock): ')

LIVE = bool(os.environ.get('ANTHROPIC_API_KEY'))
MODEL = 'claude-sonnet-4-6'  # fast + cheap for a workshop; opus-4-8 for harder reasoning
print('LIVE mode' if LIVE else 'OFFLINE mock mode (no key) — agent loop will be simulated')

LIVE mode


## Step 2 — A tiny FastAPI backend (the 'external API')

This is the kind of service your agent does **not** control — it just calls it.
We expose one route, then wrap the app in a `TestClient` so calls run in-process.

In [3]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()

_ORDERS = {
    'A1001': {'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped',  'total': 129.0},
    'A1002': {'id': 'A1002', 'item': 'USB-C hub',          'qty': 2, 'status': 'processing','total': 58.0},
    'A1003': {'id': 'A1003', 'item': '4K monitor',         'qty': 1, 'status': 'delivered', 'total': 410.0},
}

@app.get('/orders/{order_id}')
def get_order(order_id: str):
    o = _ORDERS.get(order_id.upper())
    if not o:
        raise HTTPException(status_code=404, detail='order not found')
    return o

client = TestClient(app)
print(client.get('/orders/A1001').json())

{'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped', 'total': 129.0}


## Step 3 — A Redis memory layer

Two responsibilities, the way the deck framed memory:

- **Short-term** — the running conversation, stored as a Redis **list** per session (`hist:<session>`).
- **Long-term** — durable **facts** about the user, stored in a Redis **hash** (`facts:<session>`), with an optional TTL.

`fakeredis` implements the same commands, so this class is unchanged when you move to real Redis.

In [4]:
import json, time, fakeredis

class RedisMemory:
    def __init__(self, r, session_id: str, history_limit: int = 40):
        self.r = r
        self.sid = session_id
        self.history_limit = history_limit
        self.h_key = f'hist:{session_id}'
        self.f_key = f'facts:{session_id}'

    # ---- short-term: conversation turns ----
    def append_turn(self, role: str, content):
        self.r.rpush(self.h_key, json.dumps({'role': role, 'content': content}))
        self.r.ltrim(self.h_key, -self.history_limit, -1)  # keep only the tail

    def load_history(self):
        return [json.loads(x) for x in self.r.lrange(self.h_key, 0, -1)]

    # ---- long-term: durable facts ----
    def set_fact(self, key: str, value: str, ttl_seconds: int | None = None):
        self.r.hset(self.f_key, key, value)
        if ttl_seconds:
            self.r.expire(self.f_key, ttl_seconds)

    def get_fact(self, key: str):
        v = self.r.hget(self.f_key, key)
        return v.decode() if isinstance(v, bytes) else v

    def all_facts(self):
        return {k.decode(): v.decode() for k, v in self.r.hgetall(self.f_key).items()}

r = fakeredis.FakeStrictRedis()
mem = RedisMemory(r, session_id='demo-user')
mem.set_fact('name', 'Asha')
mem.append_turn('user', 'hello')
print('facts:', mem.all_facts())
print('history:', mem.load_history())

facts: {'name': 'Asha'}
history: [{'role': 'user', 'content': 'hello'}]


## Step 4 — Declare the tools (JSON schemas)

Three client tools. Names are **namespaced by purpose** and every field is described —
the description *is* the prompt the model reads when deciding how to call.
`order_id` uses a `pattern` so the model returns well-formed IDs.

In [5]:
TOOLS = [
    {
        'name': 'get_order',
        'description': 'Look up a customer order by its ID and return item, quantity, status and total.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'order_id': {'type': 'string', 'description': "Order ID like 'A1001'.", 'pattern': '^[Aa][0-9]{4}$'}
            },
            'required': ['order_id'],
        },
    },
    {
        'name': 'remember_fact',
        'description': 'Persist a durable fact about the user (e.g. shipping preference) for future turns.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'key':   {'type': 'string', 'description': 'Short fact key, e.g. "shipping_pref".'},
                'value': {'type': 'string', 'description': 'The fact value to store.'},
            },
            'required': ['key', 'value'],
        },
    },
    {
        'name': 'recall_fact',
        'description': 'Retrieve a previously stored fact about the user by key. Returns empty if unknown.',
        'input_schema': {
            'type': 'object',
            'properties': {'key': {'type': 'string', 'description': 'The fact key to look up.'}},
            'required': ['key'],
        },
    },
]
print(len(TOOLS), 'tools declared')

3 tools declared


## Step 5 — A dispatch map: tool name → Python function

This is the boundary between *the model's intent* and *your code*. Each function returns a
string (what the model will read back). Errors are **returned**, not raised, so the agent can recover —
that maps to `is_error: true` on the `tool_result` block in Step 6.

In [6]:
def tool_get_order(order_id: str):
    resp = client.get(f'/orders/{order_id}')
    if resp.status_code == 404:
        return {'error': f'No order {order_id} found.'}
    resp.raise_for_status()
    return resp.json()

def tool_remember_fact(key: str, value: str):
    mem.set_fact(key, value)
    return {'ok': True, 'stored': {key: value}}

def tool_recall_fact(key: str):
    v = mem.get_fact(key)
    return {'key': key, 'value': v} if v is not None else {'key': key, 'value': None}

DISPATCH = {
    'get_order': tool_get_order,
    'remember_fact': tool_remember_fact,
    'recall_fact': tool_recall_fact,
}

def run_tool(name, args):
    fn = DISPATCH.get(name)
    if fn is None:
        return {'error': f'unknown tool {name}'}, True
    try:
        out = fn(**args)
        is_err = isinstance(out, dict) and 'error' in out
        return out, is_err
    except Exception as e:
        return {'error': repr(e)}, True

print(run_tool('get_order', {'order_id': 'A1002'}))
print(run_tool('get_order', {'order_id': 'A9999'}))

({'id': 'A1002', 'item': 'USB-C hub', 'qty': 2, 'status': 'processing', 'total': 58.0}, False)
({'error': 'No order A9999 found.'}, True)


## Step 6 — The agent loop (driven by `stop_reason`)

This is the heart of client-side tool use:

1. send messages + tool definitions,
2. if `stop_reason == 'tool_use'`: run **every** `tool_use` block, append a single user message
   containing one `tool_result` per call (matched by `tool_use_id`), and loop,
3. stop at `stop_reason == 'end_turn'` and return the text.

If there's no API key we simulate one scripted tool call so the notebook still runs end-to-end.

In [7]:
import json

SYSTEM = (
    'You are an order-support assistant. Use get_order for any order question. '
    'Use remember_fact / recall_fact to keep durable user preferences across turns. '
    'Be concise.'
)

def agent_turn(user_text, max_steps=6, verbose=True):
    mem.append_turn('user', user_text)
    messages = mem.load_history()

    if not LIVE:
        # ---- offline mock: pretend the model asked for get_order once ----
        if verbose: print('… (mock) model requests get_order A1001')
        out, _ = run_tool('get_order', {'order_id': 'A1001'})
        reply = f'(mock) Order A1001 is {out.get("status", "?")}.'
        mem.append_turn('assistant', reply)
        return reply

    from anthropic import Anthropic
    clientA = Anthropic()
    for step in range(max_steps):
        resp = clientA.messages.create(
            model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages,
        )
        if resp.stop_reason == 'tool_use':
            # echo the assistant turn (text + tool_use blocks) back into the transcript
            messages.append({'role': 'assistant', 'content': [b.model_dump() for b in resp.content]})
            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    if verbose: print(f'  → tool: {block.name}({block.input})')
                    out, is_err = run_tool(block.name, block.input)
                    results.append({
                        'type': 'tool_result',
                        'tool_use_id': block.id,
                        'content': json.dumps(out),
                        'is_error': is_err,
                    })
            messages.append({'role': 'user', 'content': results})
            continue
        # end_turn
        text = ''.join(b.text for b in resp.content if b.type == 'text')
        mem.append_turn('assistant', text)
        return text
    return '(stopped: max steps reached)'

print(agent_turn('What is the status of order A1002?'))

  → tool: get_order({'order_id': 'A1002'})
Here are the details for order **A1002**:

- **Item:** USB-C Hub
- **Quantity:** 2
- **Status:** Processing
- **Total:** $58.00

Your order is currently being processed. Is there anything else I can help you with?


## Step 7 — Persistence check: memory survives across turns

Because every turn is written to Redis, a fresh `agent_turn` call still sees the history and facts.
Ask the agent to remember something, then recall it in a later turn.

In [8]:
print(agent_turn('Please remember that my shipping preference is express.'))
print('---')
print(agent_turn('What did I say my shipping preference was?'))
print('---')
print('Raw facts in Redis:', mem.all_facts())
print('History length:', len(mem.load_history()), 'turns')

  → tool: remember_fact({'key': 'shipping_pref', 'value': 'express'})
Got it! I've saved your shipping preference as **express**. I'll keep that in mind for future interactions. Is there anything else I can help you with?
---
  → tool: recall_fact({'key': 'shipping_pref'})
Your shipping preference is set to **express**. Is there anything else I can help you with?
---
Raw facts in Redis: {'name': 'Asha', 'shipping_pref': 'express'}
History length: 7 turns


## Step 8 — Multi-turn chat demo

A short scripted conversation that exercises lookup + memory together.

In [9]:
for msg in [
    'Hi, I am Asha.',
    'How much was order A1003?',
    'Remember that my budget cap is 500 dollars.',
    'Given my budget cap, was that order within it?',
]:
    print('USER:', msg)
    print('AGENT:', agent_turn(msg, verbose=False))
    print()

USER: Hi, I am Asha.
AGENT: Hi Asha! Nice to meet you! 😊 Is there anything I can help you with today, such as checking an order status or anything else?

USER: How much was order A1003?
AGENT: Order **A1003** was for a **4K Monitor** (qty: 1) and the total was **$410.00**. It has been **delivered**. Is there anything else I can help you with, Asha?

USER: Remember that my budget cap is 500 dollars.
AGENT: Done! I've saved your budget cap as **$500**. Is there anything else I can help you with, Asha?

USER: Given my budget cap, was that order within it?
AGENT: Yes! Order A1003 totaled **$410.00**, which is within your budget cap of **$500.00** — you were $90 under budget! Is there anything else I can help you with, Asha?



## Extension tasks

Pick a few — these are the assignment for this lab.

1. **Rolling summary / compaction.** When `load_history()` exceeds N turns, summarise the oldest
   turns with a cheap model call and replace them with one synthetic `assistant` summary message.
   Keep total context bounded.
2. **TTL & a `forget` tool.** Add a `forget_fact(key)` tool and give facts a TTL via `set_fact(..., ttl_seconds=...)`.
   Show that an expired fact returns `None`.
3. **Parallel lookups.** Add a `get_customer` tool and a second order endpoint, then ask a question that
   needs both — observe Claude emit **two `tool_use` blocks in one turn**. Confirm your loop answers both ids.
4. **Guarded writes / PII.** Reject `remember_fact` values that look like card numbers or emails (regex);
   return `is_error: true` with a helpful message and confirm the agent recovers.
5. **Token accounting.** Read `resp.usage` each step; print cumulative input/output tokens for a turn.
6. **Real Redis.** Replace `fakeredis.FakeStrictRedis()` with `redis.Redis.from_url(os.environ['REDIS_URL'])`
   (e.g. a free Upstash/Redis Cloud URL). The `RedisMemory` class should not change at all.

> **Stretch:** swap `TestClient` for a real deployed FastAPI URL using `httpx.Client(base_url=...)`.
> Only `tool_get_order` changes; the agent loop is untouched.

In [ ]:
# Extensions

| # | Extension | What it adds |
|---|-----------|-------------|
| E1 | Rolling summary / compaction | Bounded context via cheap summarisation |
| E2 | TTL & `forget` tool | Expiring facts + agent-callable delete |
| E3 | Parallel lookups | Two `tool_use` blocks in one turn |
| E4 | Guarded writes / PII | Regex reject + `is_error` recovery |
| E5 | Token accounting | Cumulative usage per turn |
| E6 | Real Redis (Upstash) | One-line swap proof |


## E1 — Rolling summary / compaction

When history exceeds `N` turns, the **oldest half** is summarised by a cheap Claude call
and replaced with a single synthetic `assistant` message.  
Total context stays bounded; facts and recent turns are preserved.

In [10]:
import json
from anthropic import Anthropic

COMPACTION_THRESHOLD = 6   # compact when history has more than this many turns
COMPACTION_MODEL     = 'claude-haiku-4-5-20251001'  # cheap; only for summarisation

def compact_history_if_needed(mem, verbose=True):
    """
    Summarise the oldest half of history into one synthetic assistant message
    when the total turn count exceeds COMPACTION_THRESHOLD.
    Returns True if compaction happened.
    """
    history = mem.load_history()
    if len(history) <= COMPACTION_THRESHOLD:
        return False

    half = len(history) // 2
    old_turns = history[:half]
    keep_turns = history[half:]

    # Summarise with a cheap call
    clientA = Anthropic()
    summary_prompt = (
        "Summarise the following conversation turns concisely. "
        "Preserve key facts, decisions, and any order IDs or preferences mentioned.\n\n"
        + json.dumps(old_turns, indent=2)
    )
    resp = clientA.messages.create(
        model=COMPACTION_MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": summary_prompt}],
    )
    summary_text = resp.content[0].text.strip()
    summary_msg  = {"role": "assistant", "content": f"[Conversation summary] {summary_text}"}

    # Rebuild the Redis list with: summary + kept turns
    new_history = [summary_msg] + keep_turns

    import fakeredis
    # Re-write the history key
    mem.r.delete(mem.h_key)
    for turn in new_history:
        mem.r.rpush(mem.h_key, json.dumps(turn))

    if verbose:
        print(f"  [compaction] Summarised {half} turns → 1 summary + {len(keep_turns)} kept")
        print(f"  Summary: {summary_text[:120]}…")
    return True


# ---- Demo ----
import fakeredis as fr

r_e1 = fr.FakeStrictRedis()

class RedisMemory:
    def __init__(self, r, session_id, history_limit=40):
        self.r = r; self.sid = session_id; self.history_limit = history_limit
        self.h_key = f'hist:{session_id}'; self.f_key = f'facts:{session_id}'
    def append_turn(self, role, content):
        self.r.rpush(self.h_key, json.dumps({'role': role, 'content': content}))
        self.r.ltrim(self.h_key, -self.history_limit, -1)
    def load_history(self):
        return [json.loads(x) for x in self.r.lrange(self.h_key, 0, -1)]
    def set_fact(self, key, value, ttl_seconds=None):
        self.r.hset(self.f_key, key, value)
        if ttl_seconds: self.r.expire(self.f_key, ttl_seconds)
    def get_fact(self, key):
        v = self.r.hget(self.f_key, key)
        return v.decode() if isinstance(v, bytes) else v
    def all_facts(self):
        return {k.decode(): v.decode() for k, v in self.r.hgetall(self.f_key).items()}

mem_e1 = RedisMemory(r_e1, 'e1-session')

# Populate more than COMPACTION_THRESHOLD turns
sample_turns = [
    ('user',      'Hi, I ordered A1001 last week.'),
    ('assistant', 'Hello! Order A1001 is a mechanical keyboard, shipped.'),
    ('user',      'What about A1002?'),
    ('assistant', 'Order A1002 is a USB-C hub, currently processing.'),
    ('user',      'Please remember I prefer express shipping.'),
    ('assistant', 'Noted — express shipping preference stored.'),
    ('user',      'How much was order A1003?'),
    ('assistant', 'Order A1003 was $410 for a 4K monitor, delivered.'),
]
for role, content in sample_turns:
    mem_e1.append_turn(role, content)

print(f"Before compaction: {len(mem_e1.load_history())} turns")
did_compact = compact_history_if_needed(mem_e1)
print(f"After  compaction: {len(mem_e1.load_history())} turns")
print("\nFull history after compaction:")
for t in mem_e1.load_history():
    print(f"  [{t['role']}] {str(t['content'])[:80]}")


Before compaction: 8 turns
  [compaction] Summarised 4 turns → 1 summary + 4 kept
  Summary: # Conversation Summary

**Orders Discussed:**
- **A1001** (Mechanical keyboard) - Shipped
- **A1002** (USB-C hub) - Curr…
After  compaction: 5 turns

Full history after compaction:
  [assistant] [Conversation summary] # Conversation Summary

**Orders Discussed:**
- **A1001**
  [user] Please remember I prefer express shipping.
  [assistant] Noted — express shipping preference stored.
  [user] How much was order A1003?
  [assistant] Order A1003 was $410 for a 4K monitor, delivered.


## E2 — TTL & a `forget` tool

- `forget_fact(key)` deletes a single fact from Redis.
- `set_fact` already accepts `ttl_seconds`; here we demonstrate it expiring.
- The agent gets a new `forget_fact` tool so it can respond to "please forget my X".

In [11]:
import time, fakeredis as fr

r_e2 = fr.FakeStrictRedis()
mem_e2 = RedisMemory(r_e2, 'e2-session')

# --- extend RedisMemory with delete ----------------------------------------
def forget_fact(mem, key: str):
    mem.r.hdel(mem.f_key, key)

# --- new tool schema --------------------------------------------------------
FORGET_TOOL = {
    'name': 'forget_fact',
    'description': 'Delete a previously stored fact about the user.',
    'input_schema': {
        'type': 'object',
        'properties': {'key': {'type': 'string', 'description': 'The fact key to delete.'}},
        'required': ['key'],
    },
}

def tool_forget_fact(key: str):
    forget_fact(mem_e2, key)
    return {'ok': True, 'deleted': key}

# --- TTL demonstration (fakeredis honours expire) ---------------------------
print("=== TTL demo ===")
mem_e2.set_fact('promo_code', 'SAVE10', ttl_seconds=2)
print("Immediately:", mem_e2.get_fact('promo_code'))   # 'SAVE10'
time.sleep(3)
print("After 3 s :", mem_e2.get_fact('promo_code'))    # None — expired

# --- forget_fact demo -------------------------------------------------------
print("\n=== forget_fact demo ===")
mem_e2.set_fact('shipping_pref', 'express')
mem_e2.set_fact('budget_cap',    '500')
print("Before forget:", mem_e2.all_facts())

result = tool_forget_fact('budget_cap')
print("Tool returned:", result)
print("After  forget:", mem_e2.all_facts())


=== TTL demo ===
Immediately: SAVE10
After 3 s : None

=== forget_fact demo ===
Before forget: {'shipping_pref': 'express', 'budget_cap': '500'}
Tool returned: {'ok': True, 'deleted': 'budget_cap'}
After  forget: {'shipping_pref': 'express'}


## E3 — Parallel lookups (two `tool_use` blocks in one turn)

We add a `get_customer` tool backed by a new FastAPI endpoint.  
Ask a question that requires **both** an order and a customer lookup in one shot —  
Claude will emit two `tool_use` blocks simultaneously, and the loop answers both.

In [12]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
import json, fakeredis as fr

# ── extended FastAPI app ──────────────────────────────────────────────────────
app_e3 = FastAPI()

_ORDERS = {
    'A1001': {'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped',   'total': 129.0, 'customer_id': 'C001'},
    'A1002': {'id': 'A1002', 'item': 'USB-C hub',           'qty': 2, 'status': 'processing','total': 58.0,  'customer_id': 'C002'},
    'A1003': {'id': 'A1003', 'item': '4K monitor',          'qty': 1, 'status': 'delivered', 'total': 410.0, 'customer_id': 'C001'},
}
_CUSTOMERS = {
    'C001': {'id': 'C001', 'name': 'Asha Mehta',   'tier': 'gold',   'email': 'asha@example.com'},
    'C002': {'id': 'C002', 'name': 'Ravi Sharma',  'tier': 'silver', 'email': 'ravi@example.com'},
}

@app_e3.get('/orders/{order_id}')
def get_order_e3(order_id: str):
    o = _ORDERS.get(order_id.upper())
    if not o: raise HTTPException(404, 'order not found')
    return o

@app_e3.get('/customers/{customer_id}')
def get_customer_e3(customer_id: str):
    c = _CUSTOMERS.get(customer_id.upper())
    if not c: raise HTTPException(404, 'customer not found')
    return c

client_e3 = TestClient(app_e3)

# ── tools ────────────────────────────────────────────────────────────────────
TOOLS_E3 = [
    {
        'name': 'get_order',
        'description': 'Look up an order by ID. Returns item, qty, status, total and customer_id.',
        'input_schema': {'type': 'object', 'properties': {
            'order_id': {'type': 'string', 'description': "Order ID like 'A1001'.", 'pattern': '^[Aa][0-9]{4}$'}
        }, 'required': ['order_id']},
    },
    {
        'name': 'get_customer',
        'description': 'Look up a customer by customer_id. Returns name, tier and email.',
        'input_schema': {'type': 'object', 'properties': {
            'customer_id': {'type': 'string', 'description': "Customer ID like 'C001'."}
        }, 'required': ['customer_id']},
    },
]

def run_tool_e3(name, args):
    if name == 'get_order':
        r = client_e3.get(f'/orders/{args["order_id"]}')
        return (r.json(), r.status_code != 200)
    if name == 'get_customer':
        r = client_e3.get(f'/customers/{args["customer_id"]}')
        return (r.json(), r.status_code != 200)
    return ({'error': f'unknown tool {name}'}, True)

# ── agent loop for E3 ────────────────────────────────────────────────────────
r_e3 = fr.FakeStrictRedis()
mem_e3 = RedisMemory(r_e3, 'e3-session')

LIVE = bool(__import__('os').environ.get('ANTHROPIC_API_KEY'))

def agent_e3(user_text, max_steps=6):
    mem_e3.append_turn('user', user_text)
    messages = mem_e3.load_history()
    if not LIVE:
        # Offline mock: simulate parallel tool calls
        print('  (mock) Parallel tools: get_order(A1001) + get_customer(C001)')
        o, _ = run_tool_e3('get_order',    {'order_id': 'A1001'})
        c, _ = run_tool_e3('get_customer', {'customer_id': 'C001'})
        reply = f'(mock) Order {o["id"]} ({o["item"]}) for customer {c["name"]} (tier: {c["tier"]}).'
        mem_e3.append_turn('assistant', reply)
        return reply

    from anthropic import Anthropic
    clientA = Anthropic()
    SYSTEM_E3 = (
        'You are an order-support assistant. '
        'Use get_order and get_customer to answer questions. '
        'When a question needs both, call them in the same turn.'
    )
    for _ in range(max_steps):
        resp = clientA.messages.create(
            model='claude-sonnet-4-6', max_tokens=1024,
            system=SYSTEM_E3, tools=TOOLS_E3, messages=messages,
        )
        if resp.stop_reason == 'tool_use':
            messages.append({'role': 'assistant', 'content': [b.model_dump() for b in resp.content]})
            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    print(f'  → tool: {block.name}({block.input})')
                    out, is_err = run_tool_e3(block.name, block.input)
                    results.append({'type': 'tool_result', 'tool_use_id': block.id,
                                    'content': json.dumps(out), 'is_error': is_err})
            messages.append({'role': 'user', 'content': results})
            continue
        text = ''.join(b.text for b in resp.content if b.type == 'text')
        mem_e3.append_turn('assistant', text)
        return text
    return '(max steps)'

print("USER: Who placed order A1001, and what is their customer tier?")
print("AGENT:", agent_e3("Who placed order A1001, and what is their customer tier?"))


USER: Who placed order A1001, and what is their customer tier?
  → tool: get_order({'order_id': 'A1001'})
  → tool: get_customer({'customer_id': 'C001'})
AGENT: Here are the details for order **A1001**:

- **Customer Name:** Asha Mehta
- **Customer ID:** C001
- **Customer Tier:** Gold

Asha is a **Gold-tier** customer who placed an order for a Mechanical Keyboard (currently **shipped**) totaling **$129.00**.


In [ ]:
## E4 — Guarded writes / PII filtering

`remember_fact` now rejects values that look like:
- Credit/debit card numbers (13–19 digits, optionally space/dash-separated)
- Email addresses

The tool returns `is_error: true` with a helpful explanation so the agent can recover gracefully.

In [13]:
import re, fakeredis as fr

r_e4 = fr.FakeStrictRedis()
mem_e4 = RedisMemory(r_e4, 'e4-session')

# ── PII patterns ────────────────────────────────────────────────────────────
_CARD_RE  = re.compile(r'\b(?:\d[ -]?){13,19}\b')
_EMAIL_RE = re.compile(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}')

def _contains_pii(value: str) -> str | None:
    """Return a reason string if PII detected, else None."""
    if _CARD_RE.search(value):
        return 'card number'
    if _EMAIL_RE.search(value):
        return 'email address'
    return None

def tool_remember_fact_guarded(key: str, value: str):
    reason = _contains_pii(value)
    if reason:
        return {
            'error': (
                f'Refused: the value looks like a {reason}. '
                'I cannot store PII. Please ask the user to rephrase without sensitive data.'
            )
        }
    mem_e4.set_fact(key, value)
    return {'ok': True, 'stored': {key: value}}

# ── Demo ────────────────────────────────────────────────────────────────────
tests = [
    ('shipping_pref',  'express'),
    ('card_number',    '4111 1111 1111 1111'),   # card → blocked
    ('contact',        'user@example.com'),       # email → blocked
    ('budget_cap',     '750 dollars'),            # safe
]
for key, val in tests:
    result = tool_remember_fact_guarded(key, val)
    status = '✗ BLOCKED' if 'error' in result else '✓ stored '
    print(f'{status}  remember_fact({key!r}, {val!r})')
    if 'error' in result:
        print(f'          → {result["error"]}')

print("\nFacts stored:", mem_e4.all_facts())


✓ stored   remember_fact('shipping_pref', 'express')
✗ BLOCKED  remember_fact('card_number', '4111 1111 1111 1111')
          → Refused: the value looks like a card number. I cannot store PII. Please ask the user to rephrase without sensitive data.
✗ BLOCKED  remember_fact('contact', 'user@example.com')
          → Refused: the value looks like a email address. I cannot store PII. Please ask the user to rephrase without sensitive data.
✓ stored   remember_fact('budget_cap', '750 dollars')

Facts stored: {'shipping_pref': 'express', 'budget_cap': '750 dollars'}


## E5 — Token accounting

We wrap the agent loop so it reads `resp.usage` at every step and prints
**cumulative** input + output tokens after each full turn.

In [14]:
import json, fakeredis as fr
from dataclasses import dataclass, field

@dataclass
class TokenLedger:
    input_tokens:  int = 0
    output_tokens: int = 0
    steps:         int = 0

    def add(self, usage):
        self.input_tokens  += usage.input_tokens
        self.output_tokens += usage.output_tokens
        self.steps         += 1

    def report(self):
        total = self.input_tokens + self.output_tokens
        print(f'  [tokens] steps={self.steps} '
              f'in={self.input_tokens} out={self.output_tokens} total={total}')


r_e5 = fr.FakeStrictRedis()
mem_e5 = RedisMemory(r_e5, 'e5-session')

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app_e5 = FastAPI()
_ORDERS_E5 = {
    'A1001': {'id': 'A1001', 'item': 'Mechanical keyboard', 'status': 'shipped',   'total': 129.0},
    'A1002': {'id': 'A1002', 'item': 'USB-C hub',           'status': 'processing','total': 58.0},
}
@app_e5.get('/orders/{order_id}')
def _get(order_id: str):
    o = _ORDERS_E5.get(order_id.upper())
    if not o: raise HTTPException(404, 'not found')
    return o
client_e5 = TestClient(app_e5)

TOOLS_E5 = [{
    'name': 'get_order',
    'description': 'Look up a customer order by ID.',
    'input_schema': {'type': 'object',
        'properties': {'order_id': {'type': 'string', 'pattern': '^[Aa][0-9]{4}$'}},
        'required': ['order_id']},
}]

LIVE = bool(__import__('os').environ.get('ANTHROPIC_API_KEY'))

def agent_turn_e5(user_text, max_steps=6):
    """Agent loop that prints token usage per step and cumulatively."""
    mem_e5.append_turn('user', user_text)
    messages = mem_e5.load_history()
    ledger = TokenLedger()

    if not LIVE:
        print('  (mock) token accounting skipped in offline mode')
        reply = '(mock) Order A1001 is shipped for $129.'
        mem_e5.append_turn('assistant', reply)
        return reply

    from anthropic import Anthropic
    clientA = Anthropic()
    SYSTEM_E5 = 'You are an order assistant. Use get_order for order questions. Be concise.'
    for step in range(max_steps):
        resp = clientA.messages.create(
            model='claude-sonnet-4-6', max_tokens=512,
            system=SYSTEM_E5, tools=TOOLS_E5, messages=messages,
        )
        ledger.add(resp.usage)
        print(f'  step {step+1}: stop_reason={resp.stop_reason}', end='  ')
        ledger.report()

        if resp.stop_reason == 'tool_use':
            messages.append({'role': 'assistant', 'content': [b.model_dump() for b in resp.content]})
            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    r = client_e5.get(f'/orders/{block.input["order_id"]}')
                    results.append({'type': 'tool_result', 'tool_use_id': block.id,
                                    'content': json.dumps(r.json()), 'is_error': r.status_code != 200})
            messages.append({'role': 'user', 'content': results})
            continue
        text = ''.join(b.text for b in resp.content if b.type == 'text')
        mem_e5.append_turn('assistant', text)
        print(f'\nFinal reply: {text}')
        return text
    return '(max steps)'

print("USER: What is the status and total for order A1001?")
agent_turn_e5("What is the status and total for order A1001?")


USER: What is the status and total for order A1001?
  step 1: stop_reason=tool_use    [tokens] steps=1 in=611 out=58 total=669
  step 2: stop_reason=end_turn    [tokens] steps=2 in=1321 out=94 total=1415

Final reply: Order **A1001** (Mechanical Keyboard) has the following details:
- **Status:** Shipped
- **Total:** $129.00


'Order **A1001** (Mechanical Keyboard) has the following details:\n- **Status:** Shipped\n- **Total:** $129.00'

## E6 — Real Redis (Upstash / Redis Cloud)

The `RedisMemory` class is **unchanged**.  
The only edit is a one-liner swap of the connection in the setup cell.

```
# Before (fakeredis, in-process):
r = fakeredis.FakeStrictRedis()

# After (real Redis — e.g. Upstash free tier):
import redis, os
r = redis.Redis.from_url(os.environ['REDIS_URL'], decode_responses=False)
```

Set the `REDIS_URL` secret in Colab's Secrets panel (🔑) and run the cell below.
The rest of the notebook — `RedisMemory`, the agent loop, every tool — is identical.

> **Stretch:** swap `TestClient` for a real deployed FastAPI URL the same way:
> ```python
> import httpx
> http_client = httpx.Client(base_url=os.environ['FASTAPI_URL'])
> # then replace client.get(...) → http_client.get(...)
> ```
> Only `tool_get_order` changes; the agent loop is untouched.


In [17]:
!pip install upstash_redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.0 MB/s eta 0:00:00


In [20]:
import os
from upstash_redis import Redis

# Initialize the Upstash client directly
r_live = Redis(url="https://legal-termite-72897.upstash.io", token="gQAAAAAAARzBAAIgcDJjZmYzMzZlYmQxNTg0MzQ0OGNlNTA4ZDgwNDM4NDY4ZQ")

if r_live:
    # Upstash Redis client has a slightly different API than redis-py,
    # but we can wrap it or use it if the RedisMemory class is compatible.
    # For this workshop, we'll ensure mem_live uses the upstash client.
    mem_live = RedisMemory(r_live, session_id='live-session')

    # Smoke-test
    mem_live.set_fact('ping', 'pong')
    print('Connected to real Redis (Upstash). ping →', mem_live.get_fact('ping'))
    mem_live.append_turn('user', 'hello from real Redis')
    print('History:', mem_live.load_history())
else:
    print('Redis connection failed.')

Connected to real Redis (Upstash). ping → pong
History: [{'role': 'user', 'content': 'hello from real Redis'}]
